# Sales Analysis

**InsightForge Business Analytics Studio — Python Analyst Toolkit**

## Purpose
Explore revenue, order volume, average order value (AOV), top products/categories, and profit margin trends from the shared `orders.csv` schema.

## Methodology
- Data is loaded and cleaned with `clean_data.load_csv` / `clean_data.clean_dataframe` (exact duplicates removed, invalid negative values nulled, missing/outliers reported but not silently dropped).
- Trends are aggregated to monthly granularity for readability over an 18-month history.
- AOV = total revenue / total order count within each period.
- Profit margin = total profit / total revenue within each period.

## Assumptions
- `revenue`, `cost`, `profit`, `quantity` are treated as authoritative once cleaned.
- Orders with `status` of `cancelled` or `refunded` are still included in revenue/profit totals as recorded in the data (no order-status filtering) -- if your own export distinguishes these, filter by `status` before running this notebook for a completed-orders-only view.

## Limitations
- This is a synthetic sample dataset; trends here are illustrative, not real business signal.
- No inflation/price-index adjustment is applied to revenue over time.
- Analyses that depend on optional columns (e.g. `discount`) are skipped with a clear message if the column isn't present in your dataset, rather than showing misleading zeros.

In [1]:
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

TOOLKIT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, TOOLKIT_DIR)

from clean_data import load_csv, clean_dataframe

OUTPUT_DIR = os.path.join(TOOLKIT_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

orders_path = os.path.join(TOOLKIT_DIR, "sample_data", "orders.csv")
raw_orders = load_csv(orders_path)
orders, cleaning_summary = clean_dataframe(raw_orders)
print(cleaning_summary)

Data Cleaning Summary

- Removed 11 exact duplicate row(s) (kept first occurrence).
- No invalid negative values found in quantity/revenue/cost/discount/rating.
- Missing values detected (left as-is; not auto-filled to avoid inventing data):
    - 'campaign': 3831 missing (60.6% of rows)
    - 'region': 6 missing (0.1% of rows)
    - 'marketingSource': 4 missing (0.1% of rows)
    - 'discount': 3 missing (0.0% of rows)
    - 'channel': 2 missing (0.0% of rows)
- Statistical outliers detected via IQR method (reported only, not removed):
    - 'quantity': 201 value(s) outside [-0.5, 3.5]
    - 'revenue': 580 value(s) outside [-39.06, 185.38]
    - 'cost': 507 value(s) outside [-20.03, 94.18]
    - 'profit': 442 value(s) outside [-30.52, 105.58]
    - 'discount': 615 value(s) outside [-0.08, 0.12]
  Outliers are left in the cleaned data by default -- in real business data these are often legitimate large orders, not errors. Review manually before excluding.

Rows before cleaning: 6326
Row

## Revenue, Orders, and AOV by Month

In [2]:
required_cols = {"date", "revenue"}
if not required_cols.issubset(orders.columns):
    print(f"Unavailable: missing column(s) {sorted(required_cols - set(orders.columns))}")
else:
    monthly = orders.copy()
    monthly["month"] = monthly["date"].dt.to_period("M").dt.to_timestamp()
    monthly_summary = monthly.groupby("month").agg(
        totalRevenue=("revenue", "sum"),
        orderCount=("orderId", "count"),
    ).reset_index()
    monthly_summary["aov"] = monthly_summary["totalRevenue"] / monthly_summary["orderCount"]
    display(monthly_summary)

,month,totalRevenue,orderCount,aov
0,2024-01-01,16556.80,175,94.610286
1,2024-02-01,15156.58,167,90.757964
2,2024-03-01,15137.97,179,84.569665
3,2024-04-01,20396.73,228,89.459342
4,2024-05-01,28224.70,309,91.342071
5,2024-06-01,41048.87,399,102.879373
6,2024-07-01,44883.36,453,99.080265
7,2024-08-01,36488.31,408,89.432132
8,2024-09-01,32522.36,343,94.817376
9,2024-10-01,30839.22,309,99.803301


In [3]:
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
axes[0].plot(monthly_summary["month"], monthly_summary["totalRevenue"], marker="o", color="#2a5599")
axes[0].set_title("Monthly Revenue")
axes[0].set_ylabel("Revenue ($)")

axes[1].plot(monthly_summary["month"], monthly_summary["orderCount"], marker="o", color="#3c9c6a")
axes[1].set_title("Monthly Order Count")
axes[1].set_ylabel("Orders")

axes[2].plot(monthly_summary["month"], monthly_summary["aov"], marker="o", color="#c96a2e")
axes[2].set_title("Monthly Average Order Value (AOV)")
axes[2].set_ylabel("AOV ($)")
axes[2].set_xlabel("Month")

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "01_monthly_trends.png"), dpi=150)
plt.close(fig)
print("Saved output/01_monthly_trends.png")

Saved output/01_monthly_trends.png


## Top Products and Categories

In [4]:
if "product" not in orders.columns or "revenue" not in orders.columns:
    print("Unavailable: 'product' or 'revenue' column not present.")
else:
    top_products = orders.groupby("product")["revenue"].sum().sort_values(ascending=False).head(10)
    fig, ax = plt.subplots(figsize=(9, 5))
    top_products.sort_values().plot(kind="barh", ax=ax, color="#2a5599")
    ax.set_title("Top 10 Products by Revenue")
    ax.set_xlabel("Revenue ($)")
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "01_top_products.png"), dpi=150)
    plt.close(fig)
    top_products.to_csv(os.path.join(OUTPUT_DIR, "01_top_products.csv"))
    print("Saved output/01_top_products.png and output/01_top_products.csv")
    display(top_products)

Saved output/01_top_products.png and output/01_top_products.csv


product
Bamboo Cutting Board    27952.08
Resistance Bands Set    27610.12
Gift Wrap Bundle        27502.53
Ceramic Mug Set         27313.73
Mechanical Pencil       24392.19
Vitamin C Serum         23025.81
Bluetooth Earbuds       22422.06
Dog Chew Toy            21655.47
Bamboo Hairbrush        20930.18
Standing Desk Mat       20117.91
Name: revenue, dtype: float64

In [5]:
if "category" not in orders.columns or "revenue" not in orders.columns:
    print("Unavailable: 'category' or 'revenue' column not present.")
else:
    top_categories = orders.groupby("category")["revenue"].sum().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(9, 5))
    top_categories.sort_values().plot(kind="barh", ax=ax, color="#3c9c6a")
    ax.set_title("Revenue by Category")
    ax.set_xlabel("Revenue ($)")
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "01_revenue_by_category.png"), dpi=150)
    plt.close(fig)
    print("Saved output/01_revenue_by_category.png")
    display(top_categories)

Saved output/01_revenue_by_category.png


category
Office Supplies            89716.20
Electronics Accessories    84675.37
Fitness                    84032.78
Home & Kitchen             82734.17
Beauty & Personal Care     73640.20
Pet Supplies               66807.10
Stationery & Gifts         60966.92
Outdoors                   56168.90
Name: revenue, dtype: float64

## Profit Margin

In [6]:
if not {"revenue", "profit"}.issubset(orders.columns):
    print("Unavailable: 'revenue' or 'profit' column not present.")
else:
    monthly_profit = orders.copy()
    monthly_profit["month"] = monthly_profit["date"].dt.to_period("M").dt.to_timestamp()
    margin = monthly_profit.groupby("month").apply(
        lambda g: 100 * g["profit"].sum() / g["revenue"].sum() if g["revenue"].sum() else float("nan"),
        include_groups=False,
    )
    fig, ax = plt.subplots(figsize=(9, 4))
    margin.plot(marker="o", ax=ax, color="#8a3ca0")
    ax.set_title("Monthly Profit Margin (%)")
    ax.set_ylabel("Profit Margin (%)")
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "01_profit_margin.png"), dpi=150)
    plt.close(fig)
    print("Saved output/01_profit_margin.png")
    print(f"Overall profit margin: {100 * orders['profit'].sum() / orders['revenue'].sum():.1f}%")

Saved output/01_profit_margin.png
Overall profit margin: 49.3%


## Summary
This notebook wrote `01_monthly_trends.png`, `01_top_products.png` (+ `.csv`), `01_revenue_by_category.png`, and `01_profit_margin.png` to `output/`.

See the **Limitations** section at the top of this notebook -- all figures come from the synthetic sample dataset unless you point `orders_path` at your own export.